In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import os

import pandas as pd
from sqlalchemy import create_engine

## Data Loading

In [3]:
dsn = os.environ.get("CORPUS_DSN")
db_conn = create_engine(dsn)

In [6]:
df_docs = pd.read_sql("SELECT * FROM documents", con=db_conn)
print(df_docs.shape)

df_docs.head()

(1583, 5)


,id,title,source_url,published_at,word_count
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,https://basasunda.com/puisi-bahasa-sunda,2023-07-24,66
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,https://basasunda.com/puisi-bahasa-sunda,2023-07-24,94
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,https://basasunda.com/puisi-bahasa-sunda,2023-07-24,50
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,https://basasunda.com/puisi-bahasa-sunda,2023-07-24,53
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,https://basasunda.com/puisi-bahasa-sunda,2023-07-24,77


In [7]:
df_docs_raw = pd.read_sql("SELECT * FROM documents_raw", con=db_conn)
print(df_docs_raw.shape)

df_docs_raw.head()

(1583, 5)


,id,parent_id,title,content,embedding
0,51886327-762f-4477-8886-6f5091c713c7,8ea047df-1d4d-46f7-91ca-477fa1066492,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g...","[-0.5209905,-0.25163567,0.1441251,0.18745121,0..."
1,8395bb2e-4814-4265-b0a6-75118af6aeac,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...,"[-0.3721746,-0.10900261,0.06833972,0.24851488,..."
2,0e31455b-b8e1-4c36-a1e0-fe2283db2379,120e5b3e-a016-4f2a-9c87-aaded9e7195f,Hujan Poyan,"deuleu, panonpoe huhujanan boa pohaci mandi ie...","[-0.49760357,-0.091117226,-0.06612559,0.255026..."
3,35d580c9-1ae7-4bb7-8a50-2485746b084b,09f90a2c-8a4c-44f9-a3df-99cd6acae50e,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...,"[-0.44380486,-0.34206897,-0.013513703,0.242952..."
4,949ec86f-14bb-4335-a2d4-0569d5f0a582,1c332a98-82fb-4300-bb89-a7119331853b,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...,"[-0.5283572,-0.27882597,0.06334992,0.44263542,..."


## EDA

In [8]:
dupe_title = df_docs["title"].value_counts()
dupe_title[dupe_title > 1]

title
Sawér Pangantén                      5
HIJRAH NU PANUNGTUNG                 4
Katumbiri                            3
Tanah Sunda                          3
Pangabaran - 5                       3
                                    ..
USUM-USUMAN                          2
Buka Pintu                           2
ISTILAH NGASAKAN                     2
Jampé Méméh Saré                     2
Mangpaat Teknologi dina Kahirupan    2
Name: count, Length: 105, dtype: int64

In [9]:
dupe_title_url = df_docs[["title", "source_url"]].value_counts()
dupe_title_url[dupe_title_url > 1]

title                 source_url                                               
Pangabaran - 5        https://sundadigi.com/mantra/detail/88                       3
ANAK SI POLENG        https://sundadigi.com/fiksimini/chapter_fiksimini/115/939    2
Walungan Cikahuripan  https://sundadigi.com/sajak/detail/142                       2
ASBAK                 https://sundadigi.com/fiksimini/chapter_fiksimini/98/582     2
UBAR SONO             https://sundadigi.com/fiksimini/chapter_fiksimini/44/261     2
                                                                                  ..
BEURIT                https://sundadigi.com/fiksimini/chapter_fiksimini/73/1050    2
BONGAN                https://sundadigi.com/fiksimini/chapter_fiksimini/115/946    2
BANJIR                https://sundadigi.com/fiksimini/chapter_fiksimini/115/943    2
WARUNG PENGKOLAN      https://sundadigi.com/fiksimini/chapter_fiksimini/35/381     2
ABAH PALAY KURBAN     https://sundadigi.com/fiksimini/chapter_fiksimin

## Preprocessing

In [10]:
df_dedupe = df_docs.drop_duplicates(subset=["title", "source_url"], keep="last")
df_clean = df_dedupe.merge(df_docs_raw, left_on="id", right_on="parent_id")
df_clean = df_clean.drop(columns=["id_y", "title_y", "parent_id", "embedding"])
df_clean = df_clean.rename(columns={"id_x": "id", "title_x": "title"})
df_clean = df_clean[["id", "title", "content", "published_at", "word_count", "source_url"]]

df_clean

,id,title,content,published_at,word_count,source_url
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...,2023-07-24,66,https://basasunda.com/puisi-bahasa-sunda
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,eweuh deui seri nu lewih nyeri tibatan di héna...,2023-07-24,94,https://basasunda.com/puisi-bahasa-sunda
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,wanci gayuh ka peuting poék tungkeb haté lain ...,2023-07-24,50,https://basasunda.com/puisi-bahasa-sunda
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,ieu awak asa lalungsé di rasa asa carapé meure...,2023-07-24,53,https://basasunda.com/puisi-bahasa-sunda
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,naha anjeun téh poho yén mot téh dodoho datang...,2023-07-24,77,https://basasunda.com/puisi-bahasa-sunda
...,...,...,...,...,...,...
1494,0ed3b920-6d42-496d-b5dd-261f5c5c6311,Jampé Diwedak,pupur aing pupur suci nya halis katumbirén nya...,2023-12-13,37,https://sundadigi.com/mantra/detail/19
1495,0b904da0-0be0-4095-86db-afcf9539b78c,Jampé Mandi - 2,nét isun arep adus ti girang batara gangga ti ...,2023-12-19,33,https://sundadigi.com/mantra/detail/48
1496,0b6b5713-ab0a-4925-a406-97815c7b855f,Jampé Mun Aya Manuk Tuweuw Disada Peuting,gancang saré nangkuban atawa sangigir bari déd...,2024-02-01,45,https://sundadigi.com/mantra/detail/133
1497,7fe863ef-9232-41e0-8ab8-d8b1688da36e,Jampé Ngajaga Anak,sri jagat langlang mihapékeun badan kola ka be...,2024-01-08,60,https://sundadigi.com/mantra/detail/103


In [11]:
df_clean.to_json("../data/corpus.jsonl", orient="records", lines=True)